### Input and output folder structure

This notebook assumes the original RNAscope imaging data are stored locally under:

`RNAscope data/RAW/<session>/`

and writes newly converted NWB files to:

`RNAscope data/NWB/<session>/<session>.nwb`

For example, for `L1.ST8`, the input folder is expected at `RNAscope data/RAW/L1.ST8/`, and the converted NWB file is written to `RNAscope data/NWB/L1.ST8/L1.ST8.nwb`.

This conversion notebook is for local raw-to-NWB conversion. The public DANDI-downloaded NWB files used for downstream analysis are stored separately under `NWBdata/001832/`.

In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from pynwb import NWBHDF5IO
from IPython.display import display, Markdown

repo = Path(os.environ.get(
    "ANALYSIS_ROOT",
    Path.home() / "Documents" / "Repositories" / "analysis_Belal2026"
))
python_functions = repo / "Python functions"

if str(python_functions) not in sys.path:
    sys.path.insert(0, str(python_functions))

from master_RNAscope import (
    convert_rnascope_session_to_nwb,
    pretty,
    show_block,
)

raw_root = repo / "RNAscope data" / "RAW"
nwb_root = repo / "RNAscope data" / "NWB"

session = "L1.ST8"
session_out = nwb_root / session
session_out.mkdir(parents=True, exist_ok=True)

nwb_path, fields = convert_rnascope_session_to_nwb(
    session_dir=raw_root / session,
    nwb_path=session_out / f"{session}.nwb",
    timezone="America/Chicago",
    overwrite=True,
)

print(f"Wrote {nwb_path} with {len(fields)} fields")

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
    nwbfile = io.read()

    nwb_meta = {
        "session_description": nwbfile.session_description,
        "identifier": nwbfile.identifier,
        "session_start_time": nwbfile.session_start_time,
        "experiment_description": nwbfile.experiment_description,
        "experimenter": list(nwbfile.experimenter) if nwbfile.experimenter is not None else None,
        "lab": nwbfile.lab,
        "institution": nwbfile.institution,
        "protocol": nwbfile.protocol,
        "notes": nwbfile.notes,
        "keywords": list(nwbfile.keywords) if nwbfile.keywords is not None else None,
    }

    subject_meta = {}
    if nwbfile.subject is not None:
        subject_meta = {
            "subject_id": nwbfile.subject.subject_id,
            "description": nwbfile.subject.description,
            "species": nwbfile.subject.species,
            "sex": nwbfile.subject.sex,
            "genotype": nwbfile.subject.genotype,
            "age": nwbfile.subject.age,
            "strain": nwbfile.subject.strain,
        }

    custom_meta = None
    if "session_metadata_custom" in nwbfile.scratch:
        raw_custom = nwbfile.scratch["session_metadata_custom"].data
        if isinstance(raw_custom, bytes):
            raw_custom = raw_custom.decode()
        custom_meta = json.loads(raw_custom)

    counts = (
        nwbfile.processing["rnascope_analysis_metadata"]["experimenter_chrnb2_counts"]
        .to_dataframe()
        .reset_index(drop=True)
    )

display(Markdown(f"# NWB Metadata: `{nwb_path.name}`"))
show_block("NWBFile", nwb_meta)
show_block("Subject", subject_meta)
show_block("Custom", custom_meta)

display(Markdown("## Full Experimenter Counts"))
display(
    counts.sort_values(
        ["condition", "cell_type", "session", "hemisphere", "field_index", "replicate"],
        kind="stable",
    ).reset_index(drop=True)
)